# 03 · Geometria no ECG em papel

Aulas **"Visão: Geometria"** (recorte, resize, interpolação, homografia, "o documento torto volta a
ser retângulo") e **"quando a régua chega"**. O raio-X da base já vem alinhado e em 128×128, então a
geometria entra pelo outro exame do coração: o **eletrocardiograma em papel**.

O papel do ECG é uma régua impressa: quadradinho de **1 mm**, quadrado grande de **5 mm**, e o papel
corre a **25 mm/s**. Se a foto distorce a folha, a régua distorce junto, e a frequência cardíaca
medida na imagem sai errada. Neste notebook:

1. **canais de cor e HSV**: o canal que apaga a grade, num escaneado e numa foto de celular;
2. **a foto real**: a régua (pixels por mm) e quanto a frequência cardíaca depende do limiar;
3. **o gabarito**: um ECG escaneado, com frequência cardíaca medida nele;
4. **a foto torta**: o mesmo ECG fotografado em perspectiva sobre uma mesa (simulado, com gabarito);
5. **achar a folha**: Otsu → fechamento → maior contorno → 4 cantos;
6. **homografia** com quatro interpolações, medidas contra o original;
7. **recorte reto sem homografia**: o que acontece com a régua;
8. **interpolação no limite**: o experimento do documento torto da aula, com ECG real.

Imagens: amostras do [ECG-Image-Kit](https://github.com/alphanumericslab/ecg-image-kit) (licença BSD-3),
do mesmo grupo que montou a ECG-Image-Database.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.data.dados import carregar_ecg
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import geometria
from src.visao.filtros import psnr
from src.visao.metricas import dice

configurar_estilo()
SECAO = "03-geometria-ecg"

def rgb(imagem):
    return cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)

def mostrar(ax, imagem, titulo, **kw):
    if imagem.ndim == 3:
        ax.imshow(rgb(imagem))
    else:
        ax.imshow(imagem, cmap=kw.pop("cmap", "gray"), **kw)
    ax.set_title(titulo, fontsize=10)
    ax.axis("off")

## 1. Canais de cor: o canal que apaga a grade

Duas imagens de ECG: um **escaneado** limpo e uma **foto de celular** real. A imagem colorida é
**três planilhas** (no OpenCV, na ordem B, G, R). Uma grade **rosa** tem muito vermelho: no
**canal R** ela fica quase tão clara quanto o papel. O traçado preto continua escuro em todos os
canais. A pergunta é se isso vale para as duas imagens.

In [ ]:
documento = carregar_ecg(config.ECG_ESCANEADO)
documento = cv2.resize(documento, (documento.shape[1] // 2, documento.shape[0] // 2), interpolation=cv2.INTER_AREA)
ALTURA, LARGURA = documento.shape[:2]
foto_real = carregar_ecg(config.ECG_FOTO)
print("escaneado:", documento.shape, "| foto:", foto_real.shape, "| dtype:", foto_real.dtype)

fig, eixos = plt.subplots(1, 2, figsize=(16, 4.4))
mostrar(eixos[0], documento, "Escaneado (reduzido à metade)")
mostrar(eixos[1], foto_real, "Foto de celular")
salvar_figura(fig, SECAO, "duas-imagens")
plt.show()

In [ ]:
# Recortes só com grade (sem traçado), canal a canal.
recortes = {
    "escaneado": documento[200:260, 100:280],
    "foto": foto_real[470:560, 80:260],
}
fig, eixos = plt.subplots(2, 4, figsize=(15, 4.6))
contraste = {}
for linha, (nome, recorte) in enumerate(recortes.items()):
    mostrar(eixos[linha, 0], recorte, f"{nome} (RGB)")
    for ax, (canal, k) in zip(eixos[linha, 1:], [("B", 0), ("G", 1), ("R", 2)]):
        mostrar(ax, recorte[..., k], f"canal {canal}", vmin=0, vmax=255)
        contraste.setdefault(nome, {})[canal] = recorte[..., k].std()
contraste = pd.DataFrame(contraste).T.add_prefix("contraste_grade_")
salvar_figura(fig, SECAO, "canais-grade", contraste)
plt.show()
contraste.round(1)

No **escaneado**, o contraste da grade cai a um terço no canal R: a grade praticamente some.
Na **foto**, a grade saiu **cinza-arroxeada** (tinta, papel e luz do ambiente) e tem o mesmo contraste
nos três canais: nenhum canal a apaga. A cor que se vê depende da câmera e da luz, não só do objeto.

No **HSV** do escaneado, a grade aparece de outro jeito: é a única coisa **saturada** da folha
(papel branco e traçado preto têm saturação perto de zero).

In [ ]:
hsv = cv2.cvtColor(documento, cv2.COLOR_BGR2HSV)
fig, eixos = plt.subplots(1, 3, figsize=(16, 3.2))
for ax, (nome, k) in zip(eixos, [("H (matiz)", 0), ("S (saturação)", 1), ("V (brilho)", 2)]):
    mostrar(ax, hsv[..., k], f"Escaneado · {nome}")
salvar_figura(fig, SECAO, "hsv-escaneado")
plt.show()

## 2. A foto real: traçado, régua e o limite do limiar

**Traçado:** "fundo menos imagem" no canal R. Uma mediana de janela larga (31 px) estima o papel,
com a sombra junto, e ignora o traçado fino; subtraindo a imagem, só o que é mais escuro que a
vizinhança fica claro. Como a grade da foto não some em canal nenhum, quem separa é a **intensidade**:
o traçado é mais escuro que a grade, e o limiar decide onde cortar.

**Régua:** a grade se repete a cada quadrado grande. A **autocorrelação** do perfil de colunas (a
mediana de cada coluna, que ignora o traçado) tem picos a cada 1 mm, e o mais forte a cada 5 mm.

In [ ]:
px_mm_real = geometria.passo_da_grade(foto_real)
print(f"Régua da foto: {px_mm_real[0]:.2f} px/mm na horizontal, {px_mm_real[1]:.2f} px/mm na vertical")

cinza = cv2.cvtColor(foto_real, cv2.COLOR_BGR2GRAY)
perfil = np.median(cinza, axis=0).astype(np.float32)
perfil = perfil - cv2.GaussianBlur(perfil.reshape(1, -1), (0, 0), 15).ravel()
perfil -= perfil.mean()
ac = np.correlate(perfil, perfil, "full")[len(perfil) - 1 :]
ac /= ac[0]

fig, ax = plt.subplots(figsize=(15, 3.4))
ax.plot(np.arange(120), ac[:120], color=config.AZUL_COBALTO)
for k in range(1, 6):
    ax.axvline(5 * k * px_mm_real[0], color=config.CORAL, ls="--", lw=1)
ax.set(xlabel="deslocamento (pixels)", ylabel="autocorrelação",
       title=f"Perfil da grade: pico a cada quadrado grande = 5 mm = {5 * px_mm_real[0]:.1f} px")
salvar_figura(fig, SECAO, "regua-autocorrelacao", pd.DataFrame({"autocorrelacao": ac[:120]}))
plt.show()

In [ ]:
LIMIARES = [60, 100, 120, 140]
fig, eixos = plt.subplots(len(LIMIARES), 1, figsize=(14, 3.3 * len(LIMIARES)))
por_limiar = []
for ax, limiar in zip(eixos, LIMIARES):
    mascara = geometria.mascara_tracado(foto_real, limiar=limiar)
    fc = geometria.frequencia_cardiaca(mascara, float(np.mean(px_mm_real)))
    por_limiar.append({"limiar": limiar, "pixels_marcados": (mascara > 0).mean(),
                       "picos_R": len(fc["picos"]), "rr_mediano_mm": fc["rr_mm"], "fc_bpm": fc["bpm"]})
    mostrar(ax, mascara, f"limiar {limiar} · {len(fc['picos'])} picos · FC {fc['bpm']:.0f} bpm")
    ax.plot(fc["picos"], [fc["linha_base"] - fc["sinal"][p] for p in fc["picos"]], "v", color=config.CORAL, ms=7)
por_limiar = pd.DataFrame(por_limiar).set_index("limiar")
salvar_figura(fig, SECAO, "foto-limiar-fc", por_limiar)
plt.show()
por_limiar.round(3)

Com limiar baixo a grade entra na máscara; no meio, a grade sai mas as **ondas T** entram como
picos falsos e a frequência dobra; só com limiar 140 sobram os picos R, a cada ≈ 119 px. Contando a
olho na tira de ritmo: 10 picos R em ≈ 1.050 px ≈ 232 mm ≈ 9,3 s, perto de **57 bpm**, o que bate com
o limiar 140. A régua é robusta; a medida do traçado, numa foto de 4,5 px/mm, **depende do limiar**.
Para medir com gabarito, o notebook segue com o ECG **escaneado**.

## 3. O gabarito: o ECG escaneado

A **tira de ritmo** (DII longa, na base da folha) dá a frequência cardíaca: em cada coluna, a altura
do ponto mais alto do traçado acima da linha de base vira um sinal; os picos R são os picos estreitos e
altos; a mediana dos intervalos R-R, em mm, vira batimentos por minuto (FC = 60 × 25 / RR em mm).
No escaneado o canal R apaga a grade, e o limiar padrão (60) basta.

In [ ]:
px_mm_doc = float(np.mean(geometria.passo_da_grade(documento)))
mascara_doc = geometria.mascara_tracado(documento)
fc_doc = geometria.frequencia_cardiaca(mascara_doc, px_mm_doc)
print(f"Régua: {px_mm_doc:.3f} px/mm | {len(fc_doc['picos'])} picos R | RR = {fc_doc['rr_mm']:.1f} mm "
      f"= {fc_doc['rr_mm'] / 25:.3f} s | FC = {fc_doc['bpm']:.1f} bpm")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), gridspec_kw={"height_ratios": [2, 1]})
mostrar(ax1, documento, "ECG escaneado (gabarito)")
ax1.axhline(fc_doc["linha_base"], color=config.AZUL_COBALTO, lw=1, ls="--")
ax1.plot(fc_doc["picos"], [fc_doc["linha_base"] - fc_doc["sinal"][p] for p in fc_doc["picos"]], "v",
         color=config.CORAL, ms=8)
ax2.plot(fc_doc["sinal"] / px_mm_doc, color=config.AZUL_PROFUNDO, lw=1)
ax2.plot(fc_doc["picos"], fc_doc["sinal"][fc_doc["picos"]] / px_mm_doc, "v", color=config.CORAL)
ax2.set(xlabel="coluna (pixel)", ylabel="altura (mm)", title="Sinal da tira de ritmo e picos R")
salvar_figura(fig, SECAO, "gabarito-fc", pd.DataFrame({"pico_coluna": fc_doc["picos"]}))
plt.show()

## 4. A foto torta (simulada, com gabarito)

O mesmo ECG "fotografado": a folha cai em perspectiva sobre uma mesa escura (os cantos de cima mais
próximos entre si, como celular inclinado), com **sombra** de um lado e chiado do sensor. Como os
cantos verdadeiros são conhecidos, dá para medir quanto cada passo da correção acerta.

In [ ]:
rng = np.random.default_rng(config.SEMENTE)
CANTOS = np.array([[260, 170], [1430, 90], [1560, 900], [150, 820]], np.float32)
foto = geometria.simular_foto(documento, CANTOS, (1000, 1700), rng)

fig, ax = plt.subplots(figsize=(11, 6.5))
mostrar(ax, foto, "Foto simulada: perspectiva, sombra e ruído")
salvar_figura(fig, SECAO, "foto-simulada")
plt.show()

## 5. Achar a folha

A folha é clara e a mesa escura: **Otsu** separa as duas. Traçado e grade deixam furos na máscara do
papel, e o **fechamento** (dilatação seguida de erosão) os tapa. O **maior contorno** externo é a folha,
e `approxPolyDP` o simplifica até sobrar um polígono de **4 cantos**.

In [ ]:
cinza_foto = cv2.cvtColor(foto, cv2.COLOR_BGR2GRAY)
_, otsu = cv2.threshold(cv2.GaussianBlur(cinza_foto, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
fechada = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25)))
cantos = geometria.achar_papel(foto)

desenho = foto.copy()
cv2.polylines(desenho, [cantos.astype(np.int32)], True, (77, 108, 238), 4)
for x, y in cantos:
    cv2.circle(desenho, (int(x), int(y)), 14, (77, 108, 238), -1)

fig, eixos = plt.subplots(1, 3, figsize=(16, 3.6))
mostrar(eixos[0], otsu, "Otsu")
mostrar(eixos[1], fechada, "Depois do fechamento")
mostrar(eixos[2], desenho, "Contorno e 4 cantos")
erro = pd.DataFrame(np.abs(cantos - CANTOS), columns=["erro_x_px", "erro_y_px"],
                    index=["sup. esquerdo", "sup. direito", "inf. direito", "inf. esquerdo"])
salvar_figura(fig, SECAO, "achar-papel", erro)
plt.show()
erro.round(2)

## 6. Homografia e interpolação

Os 4 cantos achados vão para os 4 cantos de um retângulo do tamanho do original
(`getPerspectiveTransform` + `warpPerspective`). Cada pixel do retângulo vem de uma posição
**fracionária** da foto, e a interpolação decide o valor: vizinho mais próximo, bilinear, bicúbica
ou Lanczos.

Medidas contra o original: **PSNR** (a sombra continua na imagem endireitada e puxa o PSNR para
baixo igual para todas), **Dice** entre a máscara do traçado endireitado e a do original (a sombra
não atrapalha, porque o realce subtrai o fundo), a **régua** e a **frequência cardíaca**.

In [ ]:
miolo = (slice(20, -20), slice(20, -20))
linhas, retificadas = [], {}
for nome, metodo in geometria.INTERPOLACOES.items():
    reta = geometria.retificar(foto, cantos, LARGURA, ALTURA, metodo)
    retificadas[nome] = reta
    mascara = geometria.mascara_tracado(reta)
    px = geometria.passo_da_grade(reta)
    fc = geometria.frequencia_cardiaca(mascara, float(np.mean(px)))
    linhas.append({
        "interpolacao": nome,
        "psnr_db": psnr(documento[miolo], reta[miolo]),
        "dice_tracado": dice(mascara_doc[miolo], mascara[miolo]),
        "px_mm_horizontal": px[0], "px_mm_vertical": px[1],
        "fc_bpm": fc["bpm"],
    })
homografia = pd.DataFrame(linhas).set_index("interpolacao")
homografia.loc["original (gabarito)"] = [np.inf, 1.0, *geometria.passo_da_grade(documento), fc_doc["bpm"]]
salvar_tabela(homografia, SECAO, "homografia-interpolacoes")
homografia.round(3)

In [ ]:
zoom = (slice(30, 130), slice(880, 1080))
fig, eixos = plt.subplots(2, 5, figsize=(17, 6))
mostrar(eixos[0, 0], documento, "Original")
eixos[0, 0].add_patch(plt.Rectangle((zoom[1].start, zoom[0].start), 200, 100, fill=False, ec=config.CORAL, lw=2))
mostrar(eixos[1, 0], documento[zoom], "Original (zoom)")
for j, (nome, reta) in enumerate(retificadas.items(), start=1):
    mostrar(eixos[0, j], reta, nome)
    mostrar(eixos[1, j], reta[zoom], f"{nome} (zoom)")
salvar_figura(fig, SECAO, "homografia-zoom")
plt.show()

## 7. Recorte reto sem homografia

O atalho tentador: recortar o retângulo que envolve a folha e redimensionar para o tamanho do
original, sem corrigir a perspectiva. A régua conta a história: em cada metade da folha, quantos
pixels vale 1 mm?

In [ ]:
x, y, w, h = cv2.boundingRect(cantos.astype(np.int32))
recorte_reto = cv2.resize(foto[y : y + h, x : x + w], (LARGURA, ALTURA), interpolation=cv2.INTER_LINEAR)
reta = retificadas["Bilinear"]

regua = []
for nome, imagem in [("original", documento), ("homografia (bilinear)", reta), ("recorte reto", recorte_reto)]:
    for metade, colunas in [("esquerda", slice(0, LARGURA // 2)), ("direita", slice(LARGURA // 2, None))]:
        h_mm, v_mm = geometria.passo_da_grade(imagem[:, colunas])
        regua.append({"imagem": nome, "metade": metade, "px_mm_horizontal": h_mm, "px_mm_vertical": v_mm})
fc_recorte = geometria.frequencia_cardiaca(geometria.mascara_tracado(recorte_reto),
                                           float(np.mean(geometria.passo_da_grade(recorte_reto))))
regua = pd.DataFrame(regua).set_index(["imagem", "metade"])
salvar_tabela(regua, SECAO, "regua-por-metade")
print(f"FC no recorte reto: {fc_recorte['bpm']:.1f} bpm ({len(fc_recorte['picos'])} picos achados)")

fig, eixos = plt.subplots(1, 3, figsize=(17, 3.6))
for ax, (nome, imagem) in zip(eixos, [("Original", documento), ("Homografia (bilinear)", reta),
                                      ("Recorte reto + resize", recorte_reto)]):
    mostrar(ax, imagem, nome)
salvar_figura(fig, SECAO, "recorte-reto", regua)
plt.show()
regua.round(2)

## 8. Interpolação no limite

O experimento do documento torto da aula, com o ECG real e sem sombra: girar 18° e desfazer, com
cada interpolação. Uma volta quase não mostra diferença; **repetir** a ida e volta acumula o erro de
cada reamostragem, como acontece quando uma imagem é girada, recortada e redimensionada várias vezes
num pipeline.

In [ ]:
centro = (LARGURA / 2, ALTURA / 2)
ida = cv2.getRotationMatrix2D(centro, 18, 1.0)
volta = cv2.getRotationMatrix2D(centro, -18, 1.0)
miolo_rot = (slice(ALTURA // 4, 3 * ALTURA // 4), slice(LARGURA // 4, 3 * LARGURA // 4))

voltas = [1, 5, 10]
resultado, ultimas = [], {}
for nome, metodo in geometria.INTERPOLACOES.items():
    imagem = documento.copy()
    for n in range(1, max(voltas) + 1):
        imagem = cv2.warpAffine(imagem, ida, (LARGURA, ALTURA), flags=metodo, borderValue=(255, 255, 255))
        imagem = cv2.warpAffine(imagem, volta, (LARGURA, ALTURA), flags=metodo, borderValue=(255, 255, 255))
        if n in voltas:
            resultado.append({"interpolacao": nome, "voltas": n, "psnr_db": psnr(documento[miolo_rot], imagem[miolo_rot])})
    ultimas[nome] = imagem
rotacao = pd.DataFrame(resultado).pivot(index="interpolacao", columns="voltas", values="psnr_db")
rotacao = rotacao.loc[list(geometria.INTERPOLACOES)].add_prefix("psnr_db_")
salvar_tabela(rotacao, SECAO, "rotacao-repetida")

zoom2 = (slice(330, 400), slice(900, 1040))
fig, eixos = plt.subplots(1, 5, figsize=(17, 2.9))
mostrar(eixos[0], documento[zoom2], "Original")
for ax, (nome, imagem) in zip(eixos[1:], ultimas.items()):
    mostrar(ax, imagem[zoom2], f"{nome}\n10 voltas · {rotacao.loc[nome].iloc[-1]:.1f} dB")
salvar_figura(fig, SECAO, "rotacao-repetida", rotacao)
plt.show()
rotacao.round(2)

## Conclusões

* **Cor depende da câmera:** no escaneado, o canal R derruba o contraste da grade de 28 para 9 e
  ela some; na foto de celular a grade saiu cinza-arroxeada, com o mesmo contraste (≈ 38) nos três
  canais. O truque do canal vermelho é real, mas não é universal.
* **A régua é robusta:** a autocorrelação da grade dá ≈ 4,5 px/mm na foto real e 7,01 px/mm no
  escaneado, nas duas direções, sem depender do traçado.
* **A medida do traçado não é:** na foto real, a frequência cardíaca vai de 43 a 154 bpm conforme
  o limiar; só o limiar 140 isola os picos R (56,8 bpm, coerente com a contagem a olho, ≈ 57).
  Com 4,5 px/mm, o limiar sozinho não separa onda R de onda T.
* **Homografia devolve a régua:** na foto simulada, Otsu + fechamento + maior contorno acharam os
  4 cantos com erro de no máximo 1 px. Depois da homografia, a régua volta a 7,015 px/mm (original
  7,010) e a frequência cardíaca a 89,1 bpm (original 88,7), com qualquer interpolação.
* **Recorte reto não basta:** recortar o retângulo da folha e redimensionar deixa a grade torta;
  a "régua" mede 12 a 36 px/mm, diferente em cada metade, e nenhum pico R é achado.
* **Interpolação importa quando se repete:** numa ida e volta de 18°, o Lanczos preserva mais
  (52,7 dB) que bicúbica (47,7), bilinear (36,4) e vizinho mais próximo (33,2). Em 10 voltas, a
  **bilinear** é a que mais borra (23,2 dB), e o vizinho mais próximo serrilha uma vez e para de
  piorar (32,6 dB), porque copiar pixels não acumula média. Na homografia única, a diferença é pequena
  (Dice do traçado de 0,81 no vizinho mais próximo a 0,84 na bilinear).

**Para o trabalho:** geometria corrige a régua, mas a medida final depende de resolução e de limiar.
No raio-X, o mesmo raciocínio vale para o **índice cardiotorácico** do notebook 04: é uma razão entre
larguras e não depende da escala, mas depende de onde o limiar põe a borda do coração.